# LeetCode 2446 — Determine if Two Events Have Conflict bằng Python

Link đề: https://leetcode.com/problems/determine-if-two-events-have-conflict/ · Độ khó: **Easy** · Chủ đề: Array, String, Interval

Skeleton LeetCode:

```python
class Solution:
    def haveConflict(self, event1: List[str], event2: List[str]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **hai khoảng thời gian có chồng lấn (overlap) hay không**, trong đó hai đầu mút **đều được tính**.
2. Nắm điều kiện kinh điển: hai đoạn `[a, b]` và `[c, d]` giao nhau ⇔ `a <= d` **và** `c <= b`; hoặc phủ định: **không** giao ⇔ `b < c` hoặc `d < a`.
3. Biết rằng chuỗi `"HH:MM"` **đệm số 0, 24 giờ** có thể so sánh trực tiếp theo từ điển (lexicographic) và cho đúng thứ tự thời gian; biết cách đổi sang số phút `h * 60 + m`.
4. Viết 4 cách: tập các phút, đổi sang phút + điều kiện, so sánh chuỗi trực tiếp, và sắp xếp theo giờ bắt đầu; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: dùng `<` thay `<=` (bỏ sót chạm đầu mút), chỉ kiểm tra một chiều (bỏ sót trường hợp một sự kiện nằm trọn trong sự kiện kia), so sánh chỉ phần giờ.

## 2. Đề bài & ví dụ

Cho hai mảng chuỗi `event1` và `event2`, mỗi mảng có đúng hai phần tử `[startTime, endTime]` biểu diễn một sự kiện diễn ra từ `startTime` đến `endTime`, định dạng `"HH:MM"` theo đồng hồ **24 giờ**, và `startTime <= endTime`. Hai đầu mút đều **thuộc** sự kiện.

Hai sự kiện **xung đột (conflict)** nếu có **ít nhất một thời điểm chung**. Trả về `True` nếu hai sự kiện xung đột, ngược lại `False`.

| Ví dụ | event1 | event2 | Output | Giải thích |
|---|---|---|---|---|
| 1 | `["01:15","02:00"]` | `["02:00","03:00"]` | `True` | cùng có thời điểm `02:00` |
| 2 | `["01:00","02:00"]` | `["01:20","03:00"]` | `True` | cùng ở khoảng `01:20..02:00` |
| 3 | `["10:00","11:00"]` | `["14:00","15:00"]` | `False` | cách nhau, không chung thời điểm |

Constraints: `event1.length == event2.length == 2`; `event1[i].length == event2[i].length == 5`; `startTime1 <= endTime1`; `startTime2 <= endTime2`; mọi giá trị thời gian có định dạng `HH:MM` trong đồng hồ 24 giờ.

In [1]:
e1, e2 = ["01:15", "02:00"], ["02:00", "03:00"]
print("event1 =", e1, "| event2 =", e2)
print("độ dài chuỗi thời gian:", len(e1[0]), "| dấu ':' ở chỉ số 2:", e1[0][2])
print("so sánh chuỗi:", e1[1] >= e2[0])

event1 = ['01:15', '02:00'] | event2 = ['02:00', '03:00']
độ dài chuỗi thời gian: 5 | dấu ':' ở chỉ số 2: :
so sánh chuỗi: True


## 3. Kiến thức nền

### 3.1. Chuỗi `"HH:MM"`, tách giờ và phút

`"01:15"[:2]` là `"01"` (giờ), `"01:15"[3:]` là `"15"` (phút). `int("01")` cho `1`. Đổi sang số phút kể từ nửa đêm: `h * 60 + m`.

In [2]:
t = "01:15"
h, m = int(t[:2]), int(t[3:])
print(h, m, h * 60 + m)
h2, m2 = map(int, "23:59".split(':'))
print(h2 * 60 + m2, "<- phút cuối cùng của ngày")

1 15 75
1439 <- phút cuối cùng của ngày


### 3.2. So sánh chuỗi theo từ điển

Python so sánh chuỗi **ký tự theo ký tự** từ trái. Với `"HH:MM"` đệm số 0 và cùng độ dài, thứ tự từ điển **trùng** thứ tự thời gian: `"09:30" < "10:00"` đúng vì `'0' < '1'`. Nếu bỏ số 0 (`"9:30"`), so sánh sẽ sai (`"9:30" > "10:00"`).

In [3]:
print("09:30" < "10:00", "01:15" < "02:00", "23:59" > "00:00")
print("9:30" < "10:00", "<- sai nếu không đệm số 0")

True True True
False <- sai nếu không đệm số 0


### 3.3. Hai đoạn đóng `[a, b]` và `[c, d]` giao nhau khi nào?

Hãy nghĩ ngược: hai đoạn **không** giao nhau khi một đoạn nằm hẳn bên trái đoạn kia: `b < c` (đoạn 1 kết thúc trước khi đoạn 2 bắt đầu) hoặc `d < a`. Phủ định: giao nhau ⇔ `b >= c` **và** `d >= a`.

In [4]:
def overlap(a, b, c, d):
    return a <= d and c <= b

for seg in [(1, 5, 5, 9), (1, 5, 6, 9), (1, 10, 3, 4), (3, 4, 1, 10), (6, 9, 1, 5)]:
    print(seg, "giao nhau?", overlap(*seg))

(1, 5, 5, 9) giao nhau? True
(1, 5, 6, 9) giao nhau? False
(1, 10, 3, 4) giao nhau? True
(3, 4, 1, 10) giao nhau? True
(6, 9, 1, 5) giao nhau? False


### 3.4. `set`, `range` và giao tập hợp

Với số nguyên, đoạn đóng `[a, b]` là `set(range(a, b + 1))` (phải `+ 1` vì `range` loại phần tử cuối). Hai tập có phần tử chung ⇔ giao khác rỗng: `bool(S & T)` hoặc `not S.isdisjoint(T)`.

In [5]:
S, T = set(range(1, 6)), set(range(5, 10))
print(S & T, bool(S & T), S.isdisjoint(T))
print(set(range(1, 6)).isdisjoint(set(range(6, 10))))

{5} True False
True


### 3.5. Sắp xếp danh sách các sự kiện

`sorted(events)` sắp các cặp `[start, end]` theo `start` trước (rồi theo `end`). Sau khi sắp theo `start`, hai sự kiện xung đột ⇔ **sự kiện đến sau bắt đầu trước hoặc đúng lúc sự kiện đến trước kết thúc**: `second.start <= first.end`.

In [6]:
events = [["14:00", "15:00"], ["10:00", "11:00"]]
events.sort()
print(events)
first, second = events
print("xung đột?", second[0] <= first[1])

[['10:00', '11:00'], ['14:00', '15:00']]
xung đột? False


### 3.6. `any`, `all` và biểu thức logic ngắn gọn

`a <= d and c <= b` trả trực tiếp `bool`; không cần `if ... return True else return False`. Hãy trả thẳng biểu thức so sánh khi có thể.

In [7]:
a, b, c, d = 1, 5, 3, 9
print(a <= d and c <= b)
print(not (b < c or d < a))

True
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vẽ hai sự kiện trên trục thời gian. Chúng xung đột khi các đoạn thẳng **chạm hoặc chồng nhau**.

```
Ví dụ 1:  event1 = [01:15 ---- 02:00]
          event2 =              [02:00 ---- 03:00]      chạm ở 02:00 -> xung đột (đầu mút tính)

Ví dụ 3:  event1 = [10:00 -- 11:00]
          event2 =                       [14:00 -- 15:00]   cách xa -> không xung đột
```

Tư duy "tìm ngược": hai sự kiện **không** xung đột chỉ khi một sự kiện **kết thúc trước khi** sự kiện kia **bắt đầu**:

- `end1 < start2` (sự kiện 1 hoàn toàn trước sự kiện 2), hoặc
- `end2 < start1` (sự kiện 2 hoàn toàn trước sự kiện 1).

Nếu không rơi vào hai trường hợp đó thì có thời điểm chung. Chú ý dấu **`<` chặt** trong điều kiện "không xung đột": `end1 == start2` vẫn là xung đột vì thời điểm đó thuộc cả hai.

## 5. Cách 1 — Brute Force / baseline

Đổi mỗi thời gian sang **số phút** (`0..1439`), dựng **tập các phút** mà mỗi sự kiện chiếm, rồi kiểm tra hai tập có phần tử chung không.

Dễ hiểu đúng đề: "có ít nhất một thời điểm chung". Nút thắt: tạo tập tới `1440` phần tử chỉ để biết hai đoạn có chồng nhau hay không.

In [8]:
def to_minutes(t):
    return int(t[:2]) * 60 + int(t[3:])

class SolutionBrute:
    def haveConflict(self, event1, event2):
        minutes1 = set(range(to_minutes(event1[0]), to_minutes(event1[1]) + 1))
        minutes2 = set(range(to_minutes(event2[0]), to_minutes(event2[1]) + 1))
        return len(minutes1 & minutes2) > 0

print(SolutionBrute().haveConflict(["01:15", "02:00"], ["02:00", "03:00"]))
print(SolutionBrute().haveConflict(["01:00", "02:00"], ["01:20", "03:00"]))
print(SolutionBrute().haveConflict(["10:00", "11:00"], ["14:00", "15:00"]))

True
True
False


**Độ phức tạp:** mỗi tập tối đa `1440` phần tử → Time `O(1440)`, tức hằng số lớn; Space `O(1440)`. Nút thắt: lưu cả tập chỉ để trả lời câu hỏi "có/không".

## 6. Ý tưởng thuật toán / bước đột phá

Không cần tập hợp. Chỉ cần so sánh **bốn mốc**:

> xung đột ⇔ `start1 <= end2` **và** `start2 <= end1`.

Vì sao hai điều kiện? Mỗi điều kiện loại một kiểu "nằm hẳn một phía":

- `start1 <= end2` sai ⇒ `start1 > end2` ⇒ sự kiện 2 kết thúc **trước** khi sự kiện 1 bắt đầu.
- `start2 <= end1` sai ⇒ sự kiện 1 kết thúc **trước** khi sự kiện 2 bắt đầu.

**Quy tắc "nếu… thì…":**

- Nếu `end1 < start2` → không xung đột.
- Nếu `end2 < start1` → không xung đột.
- Còn lại → xung đột (kể cả chạm đúng một thời điểm, hoặc một sự kiện nằm trọn trong sự kiện kia).

Thêm một mẹo: chuỗi `"HH:MM"` đệm số 0 so sánh trực tiếp được, nên **không cần** đổi sang số phút.

Invariant: `max(start1, start2) <= min(end1, end2)` chính xác khi hai đoạn có điểm chung; hai điều kiện ở trên tương đương với bất đẳng thức này.

In [9]:
e1, e2 = ["01:15", "02:00"], ["02:00", "03:00"]
print("start1 <= end2 :", e1[0] <= e2[1])
print("start2 <= end1 :", e2[0] <= e1[1])
print("max(start) <= min(end):", max(e1[0], e2[0]) <= min(e1[1], e2[1]))

start1 <= end2 : True
start2 <= end1 : True
max(start) <= min(end): True


## 7. Các cách tiếp theo

Ta có 4 cách: (1) tập các phút, (2) đổi sang phút + điều kiện, (3) so sánh chuỗi trực tiếp, (4) sắp xếp theo giờ bắt đầu.

### Cách 2 — Đổi sang phút rồi dùng điều kiện giao đoạn

Chuyển bốn mốc thành số nguyên; áp công thức `s1 <= e2 and s2 <= e1`. Giữ độ an toàn nếu dữ liệu không đệm số 0.

In [10]:
class SolutionMinutes:
    def haveConflict(self, event1, event2):
        s1, e1 = to_minutes(event1[0]), to_minutes(event1[1])
        s2, e2 = to_minutes(event2[0]), to_minutes(event2[1])
        return s1 <= e2 and s2 <= e1

print(SolutionMinutes().haveConflict(["01:15", "02:00"], ["02:00", "03:00"]))
print(SolutionMinutes().haveConflict(["10:00", "11:00"], ["14:00", "15:00"]))

True
False


Độ phức tạp Cách 2: Time `O(1)`, Space `O(1)`.

### Cách 3 — So sánh chuỗi trực tiếp (cách chuẩn)

Vì định dạng cố định `HH:MM` (đệm 0, 24 giờ), so sánh từ điển đúng thứ tự thời gian. Một dòng.

In [11]:
class SolutionString:
    def haveConflict(self, event1, event2):
        return event1[0] <= event2[1] and event2[0] <= event1[1]

print(SolutionString().haveConflict(["01:00", "02:00"], ["01:20", "03:00"]))
print(SolutionString().haveConflict(["10:00", "11:00"], ["14:00", "15:00"]))

True
False


Độ phức tạp Cách 3: Time `O(1)` (so sánh chuỗi độ dài 5), Space `O(1)`.

### Cách 4 — Sắp xếp theo giờ bắt đầu rồi so sánh

Sắp hai sự kiện theo `start`; sự kiện đến sau chỉ cần so `start` của nó với `end` của sự kiện trước. Cách này là **nền tảng** của các bài "merge intervals", "meeting rooms" cho nhiều khoảng.

In [12]:
class SolutionSorted:
    def haveConflict(self, event1, event2):
        first, second = sorted([event1, event2])
        return second[0] <= first[1]

print(SolutionSorted().haveConflict(["14:00", "15:00"], ["10:00", "11:00"]))
print(SolutionSorted().haveConflict(["10:00", "11:00"], ["09:00", "12:00"]))

False
True


Độ phức tạp Cách 4: Time `O(1)` cho hai sự kiện (sắp xếp 2 phần tử), Space `O(1)`. Với `k` sự kiện: Time `O(k log k)` để sắp xếp rồi quét một lượt (theo dõi `end` lớn nhất).

## 8. TRACE từng bước

Trace Cách 3 với 3 ví dụ của đề.

| event1 | event2 | `start1 <= end2` | `start2 <= end1` | kết quả |
|---|---|---|---|---|
| `[01:15, 02:00]` | `[02:00, 03:00]` | `01:15 <= 03:00` ✓ | `02:00 <= 02:00` ✓ | `True` |
| `[01:00, 02:00]` | `[01:20, 03:00]` | `01:00 <= 03:00` ✓ | `01:20 <= 02:00` ✓ | `True` |
| `[10:00, 11:00]` | `[14:00, 15:00]` | `10:00 <= 15:00` ✓ | `14:00 <= 11:00` ✗ | `False` |

In [13]:
def trace_conflict(event1, event2):
    s1, e1 = event1
    s2, e2 = event2
    c1 = s1 <= e2
    c2 = s2 <= e1
    print(f"{event1} vs {event2}")
    print(f"  start1 <= end2 : {s1} <= {e2} -> {c1}")
    print(f"  start2 <= end1 : {s2} <= {e1} -> {c2}")
    print(f"  max(start)={max(s1, s2)}, min(end)={min(e1, e2)} -> xung đột = {c1 and c2}")
    return c1 and c2

trace_conflict(["01:15", "02:00"], ["02:00", "03:00"])
trace_conflict(["01:00", "02:00"], ["01:20", "03:00"])
trace_conflict(["10:00", "11:00"], ["14:00", "15:00"])

['01:15', '02:00'] vs ['02:00', '03:00']
  start1 <= end2 : 01:15 <= 03:00 -> True
  start2 <= end1 : 02:00 <= 02:00 -> True
  max(start)=02:00, min(end)=02:00 -> xung đột = True
['01:00', '02:00'] vs ['01:20', '03:00']
  start1 <= end2 : 01:00 <= 03:00 -> True
  start2 <= end1 : 01:20 <= 02:00 -> True
  max(start)=01:20, min(end)=02:00 -> xung đột = True
['10:00', '11:00'] vs ['14:00', '15:00']
  start1 <= end2 : 10:00 <= 15:00 -> True
  start2 <= end1 : 14:00 <= 11:00 -> False
  max(start)=14:00, min(end)=11:00 -> xung đột = False


False

## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def haveConflict(self, event1: List[str], event2: List[str]) -> bool:
        return event1[0] <= event2[1] and event2[0] <= event1[1]

print(Solution().haveConflict(["01:15", "02:00"], ["02:00", "03:00"]),
      Solution().haveConflict(["01:00", "02:00"], ["01:20", "03:00"]),
      Solution().haveConflict(["10:00", "11:00"], ["14:00", "15:00"]))

True True False


## 10. Vì sao thuật toán đúng

- **Giao khác rỗng ⇔ `max(start) <= min(end)`:** một thời điểm `t` thuộc cả hai ⇔ `start1 <= t <= end1` và `start2 <= t <= end2` ⇔ `max(start1, start2) <= t <= min(end1, end2)`. Tồn tại `t` như vậy ⇔ `max(start) <= min(end)`.
- **Khai triển:** `max(s1, s2) <= min(e1, e2)` ⇔ bốn bất đẳng thức `s1 <= e1`, `s2 <= e2` (luôn đúng theo đề), `s1 <= e2`, `s2 <= e1`. Chỉ còn hai điều kiện cần kiểm tra.
- **Đầu mút tính:** dùng `<=` vì thời điểm `end1 == start2` thuộc cả hai sự kiện.
- **So sánh chuỗi hợp lệ:** mọi chuỗi có đúng dạng `HH:MM` (đệm số 0), nên so sánh từng ký tự từ trái tương ứng so sánh giờ trước rồi phút.
- **Trường hợp lồng nhau:** nếu event2 nằm trọn trong event1 thì `s1 <= s2 <= e2 <= e1` nên cả hai điều kiện đều đúng → `True`, không bị bỏ sót.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Tập các phút | `O(1440)` (hằng số lớn) | `O(1440)` |
| 2. Đổi sang phút + điều kiện | `O(1)` | `O(1)` |
| 3. So sánh chuỗi | `O(1)` | `O(1)` |
| 4. Sắp xếp theo `start` | `O(1)` (2 sự kiện); `O(k log k)` với `k` sự kiện | `O(1)` |

Mọi so sánh chuỗi có độ dài cố định 5 nên là hằng số. Không có phân biệt average/worst. Cách 3 là gọn nhất; cách 4 mở rộng tốt nhất cho nhiều sự kiện.

## 12. Bộ test

Test 1: ví dụ đề và các trường hợp quan trọng (chạm đầu mút, lồng nhau, sát nhau một phút, cùng thời điểm).

In [15]:
impls = [SolutionBrute(), SolutionMinutes(), SolutionString(), SolutionSorted(), Solution()]
tests = [
    (["01:15", "02:00"], ["02:00", "03:00"], True),    # chạm đầu mút (bắt lỗi dùng <)
    (["01:00", "02:00"], ["01:20", "03:00"], True),    # chồng nhau một phần
    (["10:00", "11:00"], ["14:00", "15:00"], False),
    (["10:00", "11:00"], ["09:00", "12:00"], True),    # event1 nằm trọn trong event2 (bắt lỗi chỉ xét một chiều)
    (["09:00", "12:00"], ["10:00", "11:00"], True),    # event2 nằm trọn trong event1
    (["00:00", "00:00"], ["00:00", "00:00"], True),    # trùng đúng một thời điểm
    (["23:59", "23:59"], ["00:00", "23:58"], False),   # sát nhau một phút
    (["01:00", "01:30"], ["01:45", "02:00"], False),   # cùng giờ 01 nhưng không chồng (bắt lỗi chỉ xét giờ)
    (["01:00", "01:00"], ["01:01", "01:01"], False),
    (["12:00", "12:00"], ["12:00", "13:00"], True),
    (["08:30", "09:00"], ["09:01", "10:00"], False),
    (["00:00", "23:59"], ["12:34", "12:34"], True),    # cả ngày
]
for e1, e2, exp in tests:
    for im in impls:
        assert im.haveConflict(list(e1), list(e2)) is exp, (type(im).__name__, e1, e2, exp)
        assert im.haveConflict(list(e2), list(e1)) is exp, ("đối xứng", type(im).__name__, e1, e2)
print("tất cả test cố định OK (kể cả hoán đổi event1/event2):", len(tests))

tất cả test cố định OK (kể cả hoán đổi event1/event2): 12


Test 2: ngẫu nhiên có seed. Thời gian được sinh hợp lệ (giờ `0..23`, phút `0..59`, đệm số 0, `start <= end`). Oracle độc lập: đếm từng phút trong khoảng của hai sự kiện.

In [16]:
import random
random.seed(2446)

def rand_time():
    return random.randint(0, 23) * 60 + random.randint(0, 59)

def fmt(x):
    return f"{x // 60:02d}:{x % 60:02d}"

def rand_event(short):
    a = rand_time()
    b = min(1439, a + random.randint(0, 90)) if short else rand_time()
    a, b = min(a, b), max(a, b)
    return [fmt(a), fmt(b)]

def oracle(e1, e2):
    r1 = range(to_minutes(e1[0]), to_minutes(e1[1]) + 1)
    r2 = range(to_minutes(e2[0]), to_minutes(e2[1]) + 1)
    return any(m in r2 for m in r1)

trues = 0
for _ in range(4000):
    short = random.random() < 0.7
    e1, e2 = rand_event(short), rand_event(short)
    assert e1[0] <= e1[1] and e2[0] <= e2[1]
    exp = oracle(e1, e2)
    trues += exp
    for im in impls:
        assert im.haveConflict(list(e1), list(e2)) is exp, (type(im).__name__, e1, e2)
print("4000 test ngẫu nhiên OK; số test True:", trues)
assert 0 < trues < 4000

4000 test ngẫu nhiên OK; số test True: 940


Test 3: vét cạn trên lưới nhỏ — mọi cặp `(start, end)` trong một dải 6 phút (`00:00..00:05`), đảm bảo bắt hết biên và đầu mút.

In [17]:
pts = [fmt(x) for x in range(6)]
cnt = 0
for a1 in pts:
    for b1 in pts:
        if a1 > b1: continue
        for a2 in pts:
            for b2 in pts:
                if a2 > b2: continue
                e1, e2 = [a1, b1], [a2, b2]
                exp = oracle(e1, e2)
                assert all(im.haveConflict(e1, e2) is exp for im in impls), (e1, e2)
                cnt += 1
print("vét cạn", cnt, "cặp sự kiện OK")

vét cạn 441 cặp sự kiện OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `<` thay vì `<=` (bỏ sót trường hợp chạm đầu mút)

In [18]:
def wrong_strict(event1, event2):
    return event1[0] < event2[1] and event2[0] < event1[1]       # SAI: đầu mút cũng tính

e1, e2 = ["01:15", "02:00"], ["02:00", "03:00"]
print("sai :", wrong_strict(e1, e2))
print("đúng:", Solution().haveConflict(e1, e2))
assert wrong_strict(e1, e2) != Solution().haveConflict(e1, e2)

sai : False
đúng: True


Vì sao sai: `02:00` thuộc cả hai sự kiện (đề: đầu mút được tính), nhưng `02:00 < 02:00` là `False`. **Sửa:** dùng `<=`.

### Lỗi 2: chỉ kiểm tra một chiều (bỏ sót sự kiện nằm trọn trong sự kiện kia)

In [19]:
def wrong_one_side(event1, event2):
    # SAI: chỉ hỏi "start2 có nằm trong event1 không?"
    return event1[0] <= event2[0] <= event1[1]

e1, e2 = ["10:00", "11:00"], ["09:00", "12:00"]
print("sai :", wrong_one_side(e1, e2))
print("đúng:", Solution().haveConflict(e1, e2))
assert wrong_one_side(e1, e2) != Solution().haveConflict(e1, e2)

sai : False
đúng: True


Vì sao sai: event2 bắt đầu **trước** event1 nhưng bao trùm nó; `start2 = 09:00` không nằm trong event1 nên kiểm tra một chiều trả `False`. **Sửa:** kiểm tra cả hai điều kiện `s1 <= e2` và `s2 <= e1` (hoặc thử cả hai chiều).

### Lỗi 3: chỉ so sánh phần giờ

In [20]:
def wrong_hours_only(event1, event2):
    h = lambda t: int(t[:2])
    return h(event1[0]) <= h(event2[1]) and h(event2[0]) <= h(event1[1])   # SAI: bỏ qua phút

e1, e2 = ["01:00", "01:30"], ["01:45", "02:00"]
print("sai :", wrong_hours_only(e1, e2))
print("đúng:", Solution().haveConflict(e1, e2))
assert wrong_hours_only(e1, e2) != Solution().haveConflict(e1, e2)

sai : True
đúng: False


Vì sao sai: hai sự kiện cùng nằm trong giờ `01` nên giờ không phân biệt được, nhưng `01:30 < 01:45` nên chúng không chồng nhau. **Sửa:** so sánh cả giờ lẫn phút (so chuỗi đầy đủ hoặc đổi sang phút).

### Lỗi 4 (bonus): so sánh chuỗi không đệm số 0

Nếu thời gian có dạng `"9:30"` (thiếu số 0), `"9:30" > "10:00"` theo từ điển. Đề bảo đảm đủ `HH:MM`, nhưng nếu tự nhập dữ liệu hãy đệm hoặc đổi sang phút.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai khoảng có chồng nhau không" → công thức `start1 <= end2 and start2 <= end1` (đoạn **đóng**); với đoạn nửa mở `[s, e)` dùng `<` thay `<=`.
- Nhớ cách chứng minh bằng "tìm ngược": không giao ⇔ một đoạn hoàn toàn trước đoạn kia.
- Chuỗi định dạng cố định, đệm số 0 → so sánh chuỗi trực tiếp, đỡ phải parse.
- Hãy tự vẽ trục thời gian cho 3 trường hợp: tách rời, chạm, lồng nhau; kiểm tra công thức trên cả ba.
- Với nhiều sự kiện: sắp theo `start` rồi quét một lượt, so `next.start <= max_end` đã gặp.
- Luôn thử đổi chỗ `event1` và `event2` — kết quả phải **đối xứng**.

## 15. Biến thể của bài

1. Sự kiện là **nửa mở** `[start, end)` — chạm đầu mút không còn xung đột (đổi `<=` thành `<`).
2. Kiểm tra **một danh sách** sự kiện có đôi nào xung đột không (Meeting Rooms).
3. Đếm số **phút chung** của hai sự kiện (độ dài giao, bài 2409 Count Days Spent Together).
4. Gộp các khoảng chồng nhau (Merge Intervals).
5. Số phòng họp tối thiểu để không xung đột (Meeting Rooms II, dùng heap hoặc sweep line).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tập các phút | giao hai tập `{phút}` | `O(1440)` | `O(1440)` | oracle test, dễ hiểu nhất |
| 2. Đổi sang phút + điều kiện | số nguyên, `s1 <= e2 and s2 <= e1` | `O(1)` | `O(1)` | dữ liệu có thể không đệm số 0 |
| 3. So sánh chuỗi | như trên nhưng bằng chuỗi | `O(1)` | `O(1)` | **chuẩn**, ngắn nhất |
| 4. Sắp theo `start` | `second.start <= first.end` | `O(1)` / `O(k log k)` | `O(1)` | mở rộng cho nhiều sự kiện |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `haveConflict(["05:00", "06:00"], ["06:00", "07:00"])` trả về gì?

**Bài 2 (tính tay):** vẽ trục thời gian cho `["08:00","10:30"]` và `["10:31","12:00"]`; chúng có xung đột không? Vì sao?

**Bài 3 (điền TODO):** hoàn thành `no_overlap(event1, event2)` trả về `True` khi hai sự kiện **không** chồng nhau (dùng cách "một đoạn nằm hẳn trước đoạn kia").

**Bài 4 (tự viết lại):** viết `overlap_minutes(event1, event2)` trả về **số phút chung** (tính cả hai đầu, ví dụ `["01:00","02:00"]` và `["01:20","03:00"]` → `41`).

**Bài 5 (mini challenge):** viết `any_conflict(events)` nhận danh sách nhiều sự kiện và trả về `True` nếu **có ít nhất một cặp** xung đột.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().haveConflict(["05:00", "06:00"], ["06:00", "07:00"]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def no_overlap(event1, event2):
    # TODO: True nếu event1 kết thúc trước event2 bắt đầu HOẶC ngược lại
    pass

True


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** vẽ trục số; hai đoạn đóng chạm nhau tại một điểm vẫn được coi là có điểm chung.

**Mức 2 (vừa):** Bài 3: `event1[1] < event2[0] or event2[1] < event1[0]`. Bài 4: `lo = max(start1, start2)`, `hi = min(end1, end2)` (theo phút), kết quả `max(0, hi - lo + 1)`. Bài 5: sắp theo `start`, rồi so từng sự kiện với `end` lớn nhất đã gặp.

**Mức 3 (gần đáp án):** Bài 5: sắp `events` theo `start`, giữ biến `max_end` là giờ kết thúc lớn nhất đã gặp; với mỗi sự kiện mới, nếu `start <= max_end` thì trả `True`, ngược lại cập nhật `max_end = max(max_end, end)`. (Chỉ so cặp liền kề thì có thể bỏ sót khi một sự kiện rất dài bao trùm nhiều sự kiện sau.) Bài 2: `10:30 < 10:31` nên không xung đột.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: chạm đầu mút 06:00 -> True
assert Solution().haveConflict(["05:00", "06:00"], ["06:00", "07:00"]) is True

# Bài 2: 10:30 < 10:31 -> không chồng
assert Solution().haveConflict(["08:00", "10:30"], ["10:31", "12:00"]) is False

# Bài 3
def no_overlap(event1, event2):
    return event1[1] < event2[0] or event2[1] < event1[0]
assert no_overlap(["10:00", "11:00"], ["14:00", "15:00"]) is True
assert no_overlap(["01:15", "02:00"], ["02:00", "03:00"]) is False

# Bài 4
def overlap_minutes(event1, event2):
    lo = max(to_minutes(event1[0]), to_minutes(event2[0]))
    hi = min(to_minutes(event1[1]), to_minutes(event2[1]))
    return max(0, hi - lo + 1)
assert overlap_minutes(["01:00", "02:00"], ["01:20", "03:00"]) == 41
assert overlap_minutes(["10:00", "11:00"], ["14:00", "15:00"]) == 0
assert overlap_minutes(["01:15", "02:00"], ["02:00", "03:00"]) == 1

# Bài 5: sắp theo start và theo dõi end lớn nhất đã gặp
def any_conflict(events):
    events = sorted(events)
    max_end = None
    for start, end in events:
        if max_end is not None and start <= max_end:
            return True
        max_end = end if max_end is None else max(max_end, end)
    return False
assert any_conflict([["09:00", "10:00"], ["10:00", "11:00"]]) is True
assert any_conflict([["09:00", "10:00"], ["10:01", "11:00"], ["12:00", "13:00"]]) is False
assert any_conflict([["09:00", "17:00"], ["10:00", "11:00"], ["18:00", "19:00"]]) is True
assert any_conflict([["09:00", "09:30"]]) is False
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai đoạn đóng có điểm chung":**

```python
return start1 <= end2 and start2 <= end1      # đoạn đóng, đầu mút tính
# đoạn nửa mở [s, e):  start1 < end2 and start2 < end1
```

**Checklist trước khi submit:**

- [ ] Dùng `<=` (đầu mút được tính) cho đoạn đóng?
- [ ] Kiểm tra **cả hai** điều kiện (không chỉ một chiều)?
- [ ] Đã thử trường hợp chạm đầu mút, lồng nhau, sát nhau một phút?
- [ ] Kết quả đối xứng khi đổi chỗ `event1` và `event2`?
- [ ] Chuỗi `"HH:MM"` đệm số 0 nên so sánh chuỗi là hợp lệ?
- [ ] Đã đối chiếu các cách với oracle bằng test ngẫu nhiên có seed?
- [ ] Chữ ký đúng `haveConflict(self, event1: List[str], event2: List[str]) -> bool`?